# 进阶教程（一）：LangChain 核心概念深入

> 面向已掌握基础的开发者，深入 Chains / Agents / Memory / Retrievers 四大件的进阶用法。

## 本讲内容
1. **Chains**：LCEL 组合术（并行 / 动态路由 / 级联兜底）
2. **Agents**：结构化输出 + Agent 作为可组合单元
3. **Memory**：从会话记忆到跨线程长期记忆（Store）
4. **Retrievers**：多查询 / 混合检索 / 检索后压缩

# 0. 环境准备与运行说明

**前置要求：**
- 根目录 `.env` 已配置 `DEEPSEEK_API_KEY`（本教程用真实 DeepSeek，无本地降级）
- 已安装：`langchain>=1.3`、`langgraph>=1.2`、`langchain-deepseek`、`python-dotenv`
- 使用本地 `bge-small-zh-v1.5` 嵌入的章节首次运行会下载模型（约 100MB，走 hf-mirror 镜像）

**运行说明：**
- 按 cell 顺序执行；除标注外，每个示例消耗少量 API 额度（单次 < 0.01 元量级）
- 本教程面向已学完 `langchain_tutorial/` 与 `langgraph_tutorial/` 基础篇的开发者
- 涉及导入路径的坑（如 `create_agent` 在 `langchain.agents`）已在 FAQ 中汇总

In [1]:

# ========== 0. 初始化（每个 notebook 第一格） ==========
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=DeprecationWarning)

# HF 镜像必须先于任何 langchain/huggingface 导入设置（详见 rag_qa_project FAQ）
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")

ROOT = Path.cwd().parent  # advanced_tutorial 的上一级 = 项目根目录
sys.path.insert(0, str(ROOT))
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

assert os.getenv("DEEPSEEK_API_KEY"), "请先在根目录 .env 配置 DEEPSEEK_API_KEY"

# 真实 LLM：DeepSeek（本教程要求真实模型）
from langchain_deepseek import ChatDeepSeek

model = ChatDeepSeek(model="deepseek-chat", temperature=0.2)
print("模型就绪:", model.__class__.__name__)


模型就绪: ChatDeepSeek


## 1. Chains 进阶：LCEL 组合术

LCEL 的核心思想：一切皆 Runnable，用管道组合出数据流。
进阶技巧集中在三件事：**并行、动态路由、容错**。

### 1.1 RunnableParallel：并行分支与汇总

多个分支同时执行（内部线程池），最后自动合并为 dict——
这是性能优化的第一手段，也常用于 RAG 的"问题改写 + 检索"并行。

In [2]:

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

summary_chain = (
    ChatPromptTemplate.from_template("用一句话总结：{text}")
    | model | StrOutputParser()
)
keywords_chain = (
    ChatPromptTemplate.from_template("提取 3 个关键词，逗号分隔：{text}")
    | model | StrOutputParser()
)

parallel = RunnableParallel(
    summary=summary_chain,
    keywords=keywords_chain,
)

result = parallel.invoke({"text": "LangGraph 用状态图编排 LLM 工作流，"
                                  "支持循环、分支与持久化，是构建可靠 Agent 的底层引擎。"})
print("摘要:", result["summary"])
print("关键词:", result["keywords"])

摘要: LangGraph 通过状态图驱动 LLM 工作流，以循环、分支和持久化能力，为构建可靠 Agent 提供底层编排引擎。
关键词: LangGraph,状态图,Agent


**预期输出**（大意）：
```
摘要: LangGraph 以状态图方式编排 LLM 工作流……
关键词: LangGraph, 状态图, 工作流
```

两个分支并行执行，总耗时约等于较慢的那个分支，而不是两者之和。

### 1.2 动态路由：RunnableLambda 分发

按输入内容把请求分发到不同处理链——路由逻辑是纯 Python 函数，
返回的 key 决定走哪条分支。

In [3]:

from langchain_core.runnables import RunnableLambda

tech_chain = ChatPromptTemplate.from_template(
    "你是技术专家，回答：{q}") | model | StrOutputParser()
casual_chain = ChatPromptTemplate.from_template(
    "你是闲聊伙伴，轻松回答：{q}") | model | StrOutputParser()

def route(q: str) -> str:
    """路由函数：含技术关键词走 tech，否则走 casual"""
    return "tech" if any(k in q for k in ["LangChain", "LangGraph", "API", "代码"]) \
           else "casual"

router = RunnableLambda(route)
chain = (
    {"q": RunnablePassthrough()}
    | RunnableLambda(lambda x: print(f"[route] -> {route(x['q'])}") or x)
    | RunnableLambda(lambda x: (tech_chain if route(x["q"]) == "tech"
                                else casual_chain).invoke(x))
)
print(chain.invoke("LangGraph 的 checkpointer 是什么？")[:80])

[route] -> tech
## LangGraph 的 Checkpointer 是什么？

**Checkpointer** 是 LangGraph 中用于**持久化图执行状态**的核


### 1.3 级联兜底：with_fallbacks

主链路故障时自动切换备用链路——生产环境的标配。
首选链故意抛错，观察兜底生效：

In [4]:

from langchain_core.runnables import RunnableLambda

def flaky(q):
    raise ConnectionError("模拟主服务宕机")

backup_chain = ChatPromptTemplate.from_template("备用通道回答：{q}") \
    | model | StrOutputParser()

robust = (
    RunnableLambda(flaky)
    .with_fallbacks([{"q": RunnablePassthrough()} | backup_chain])
)
print(robust.invoke("什么是 RAG？")[:100])

**RAG（Retrieval-Augmented Generation，检索增强生成）** 是一种结合**信息检索**与**大语言模型（LLM）** 的AI技术架构。

简单来说，它的核心思路是：*


**要点**：`with_fallbacks` 接受 Runnable 列表，依次尝试直到成功。
典型生产组合：DeepSeek 官方 API → 中转站 → 静态回答。

## 2. Agents 进阶

### 2.1 结构化输出：response_format

让 Agent 输出 Pydantic 校验过的结构化结果，
把"自由文本 Agent"变成"可编程 API"：

In [6]:

from pydantic import BaseModel, Field
from langchain.agents import create_agent

class Answer(BaseModel):
    """带依据的回答"""
    answer: str = Field(description="最终回答")
    confidence: float = Field(description="置信度 0-1")
    needs_more_info: bool = Field(description="是否需要追问用户")

agent = create_agent(
    model=model,
    tools=[],
    system_prompt="你严谨的问答助手，回答必须给出置信度。",
    response_format=Answer,
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "地球到月球多远？"}]})
parsed = result["structured_response"]
print(type(parsed).__name__)
print("回答:", parsed.answer)
print("置信度:", parsed.confidence)

Answer
回答: 地球到月球的平均距离约为 **384,400公里**（约38.44万公里）。

具体来说：
- **平均距离**：约384,400公里
- **最近距离（近地点）**：约363,300公里
- **最远距离（远地点）**：约405,500公里

由于月球绕地球运行的轨道是椭圆形的，所以距离会有所变化。这个距离大约是地球赤道周长的9.6倍，光从地球到月球大约需要1.28秒。
置信度: 0.98


**预期输出**（大意）：
```
Answer
回答: 约 38.4 万公里
置信度: 0.98
```

`result["structured_response"]` 是 Pydantic 实例，字段类型由框架保证。

### 2.2 Agent 作为可组合单元（子代理模式）

Agent 本身是 CompiledStateGraph（Runnable），可以直接嵌入更大的图/链中——
这是多 Agent 系统的基本粒子：

In [7]:

from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """查询城市天气。Args: city: 城市名"""
    return {"北京": "晴 25C", "上海": "多云 28C"}.get(city, f"{city} 未收录")

weather_agent = create_agent(
    model=model, tools=[get_weather],
    system_prompt="你是天气助手，必须用工具查询。", name="weather_agent")

# 把 Agent 当节点用：外层链负责"意图识别后转发"
def ask_weather(question: str) -> str:
    r = weather_agent.invoke({"messages": [{"role": "user", "content": question}]})
    return r["messages"][-1].content

print(ask_weather("北京天气怎么样？"))

北京今天的天气是**晴天**，气温 **25°C**，天气不错，适合外出活动！☀️


## 3. Memory 进阶：从会话到长期记忆

| 类型 | 机制 | 生命周期 | 典型用途 |
|---|---|---|---|
| 短期记忆 | checkpointer（按 thread_id 隔离） | 单会话 | 多轮对话上下文 |
| 长期记忆 | BaseStore（按 namespace 隔离） | 跨会话/跨线程 | 用户偏好、事实沉淀 |

In [9]:

from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent

store = InMemoryStore()

# 模拟"画像采集"环节写入长期记忆（namespace 按 用户/领域 组织）
store.put(("user", "alice", "prefs"), "style", {"text": "回答保持简洁，用中文"})

# 读取记忆并注入 system prompt —— 长期记忆的标准用法
def build_agent_with_memory(user: str):
    pref = store.get(("user", user, "prefs"), "style")
    extra = pref.value["text"] if pref else ""
    return create_agent(
        model=model, tools=[], checkpointer=InMemorySaver(),
        system_prompt=f"你是助手。用户偏好：{extra}")

# 新线程中 agent 自动遵守"几天前"写入的偏好
agent_a = build_agent_with_memory("alice")
r = agent_a.invoke(
    {"messages": [{"role": "user", "content": "介绍 RAG"}]},
    config={"configurable": {"thread_id": "t1"}})
print(r["messages"][-1].content[:100])

RAG（检索增强生成，Retrieval-Augmented Generation）是一种结合**信息检索**与**大语言模型生成**的技术框架。

**核心思想**：  
在模型回答前，先从外部知识


**要点**：短期记忆换 thread_id 即失效；长期记忆跟随 **namespace**
（如 `("user", "alice", "prefs")`）跨线程、甚至跨 Agent 存活。
生产环境把 `InMemoryStore` 换成 `PostgresStore` 等持久化实现即可，接口不变。

## 4. Retrievers 进阶

In [5]:
from langchain_community.embeddings import DashScopeEmbeddings
# 准备 mini 向量库（内存 Chroma + 本地 bge 嵌入，首次运行自动下载模型）
from langchain_chroma import Chroma
from langchain_core.documents import Document
from dotenv import load_dotenv
import os
load_dotenv()


embeddings = DashScopeEmbeddings(
    model="qwen3.7-text-embedding",
    dashscope_api_key=os.getenv("DASHSCOPE_API_KEY"),
)

docs = [
    Document(page_content="LangGraph 的 checkpointer 在每个节点执行后保存状态快照，"
                          "支持断点续跑与时间旅行调试。", metadata={"topic": "langgraph"}),
    Document(page_content="DeepSeek-V3 采用 MoE 架构，671B 总参数、37B 激活参数，"
                          "在代码与数学推理上对标一线闭源模型。", metadata={"topic": "model"}),
    Document(page_content="RAG 通过检索外部知识再生成，能有效抑制大模型幻觉，"
                          "是知识密集型场景的主流方案。", metadata={"topic": "rag"}),
]

vectorstore = Chroma.from_documents(docs, embeddings, collection_name="mini")
vs_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print("向量检索:", [d.metadata["topic"] for d in vs_retriever.invoke("什么是幻觉的解药")])

向量检索: ['rag', 'rag']


### 4.1 MultiQueryRetriever：查询扩展

用户提问往往与文档措辞不一致。让 LLM 生成多个查询变体，
分别检索后合并去重——召回率显著提升：

In [10]:

from langchain_classic.retrievers.multi_query import MultiQueryRetriever

mq_retriever = MultiQueryRetriever.from_llm(
    retriever=vs_retriever, llm=model)

import logging
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

docs = mq_retriever.invoke("怎么让大模型不胡说八道")
print("多查询检索命中:", [d.metadata["topic"] for d in docs])

多查询检索命中: ['rag', 'rag']


### 4.2 EnsembleRetriever：BM25 + 向量混合检索

关键词检索（BM25）擅长精确术语，向量检索擅长语义泛化，
两者加权融合是 RAG 检索质量的性价比之王：

In [11]:

from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

bm25 = BM25Retriever.from_documents(docs, k=2)

ensemble = EnsembleRetriever(
    retrievers=[bm25, vs_retriever],   # 权重默认均分
    weights=[0.4, 0.6],
)
print("混合检索:", [d.metadata["topic"] for d in ensemble.invoke("checkpointer 断点续跑")])

混合检索: ['langgraph', 'rag']


### 4.3 检索后压缩：只把"相关句子"喂给模型

检索命中 ≠ 全文有用。用 LCEL 手写一条"检索→过滤"链，
先粗检索再让 LLM 挑出真正相关的片段，节省 token 且降噪：

In [3]:

from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
from  langchain_core.output_parsers import StrOutputParser
import os

os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")
compress_prompt = ChatPromptTemplate.from_messages([
    ("system", "从候选文档中摘出与问题直接相关的句子，没有则输出'无'。"),
    ("human", "问题：{q}\n\n候选文档：\n{docs}"),
])

def format_docs(ds):
    return "\n".join(f"[{i}] {d.page_content}" for i, d in enumerate(ds))

try:
    cross_encoder = HuggingFaceCrossEncoder(
        model_name="BAAI/bge-reranker-base"  # 中文重排序模型
    )
    reranker = CrossEncoderReranker(
        model=cross_encoder,
        top_n=3,  # 重排序后保留前 3 个
    )
    compression_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=vs_retriever,
    )

    compress_chain = (
        {"q": RunnableLambda(lambda x: x["q"]),
         "docs": RunnableLambda(lambda x: format_docs(vs_retriever.invoke(x["q"])))}
        | compress_prompt | model | StrOutputParser()
    )
    print(compress_chain.invoke({"q": "MoE 架构的参数量是多少"}))


except Exception as exc:
    print(f"重排序模型加载失败（可能未下载）: {exc}")
    print("可跳过本节，继续后续内容")
    RERANKER_AVAILABLE = False



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

重排序模型加载失败（可能未下载）: name 'vs_retriever' is not defined
可跳过本节，继续后续内容


## 5. 常见问题（FAQ）

| 问题 | 原因 | 解决 |
|---|---|---|
| `cannot import name 'EnsembleRetriever' from 'langchain_community'` | community 包已移除经典检索器 | 从 `langchain_classic.retrievers` 导入 |
| `with_structured_output` 报 NotImplementedError | Fake 模型未实现 | 自定义 Fake 时覆盖该方法（见 rag_qa_project/tests） |
| Agent 没记住上一轮对话 | 未配置 checkpointer | `create_agent(checkpointer=InMemorySaver())` + thread_id |
| 长期记忆写入了但读不到 | store 命名空间不一致 | put/get 的 namespace 与 key 必须完全一致 |
| `certificate verify failed ... huggingface.co` | HF_ENDPOINT 设置太晚 | 在任何 langchain/chromadb 导入前设置（见首格注释） |